# 🏛️ EDA — Patrimonio Cultural de Euskadi
**Fuente:** OpenData Euskadi  
**Dataset:** Espacios patrimoniales y museos registrados en Turismo Euskadi

Este análisis exploratorio cubre:
- Distribución por tipo de espacio y categoría cultural
- Análisis por provincia y municipio
- Capacidad, visitas guiadas y tiendas
- Valoraciones y popularidad
- Distribución geográfica
- Completitud del dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

PALETTE = ['#1A5276', '#1E8449', '#B7950B', '#7D3C98', '#CB4335', '#117A65', '#D35400', '#2C3E50']
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Librerías cargadas')

## 1. Carga y primera inspección del dataset

In [ ]:
df = pd.read_csv('patrimonio.csv', sep=None, engine='python', index_col=0)

print(f'📊 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'📋 Columnas: {df.columns.tolist()}')
df.head(3)

In [ ]:
df.info()

In [ ]:
df[['valoracion', 'num_resenas', 'Capacidad', 'Visita Guiada', 'Tienda']].describe().round(2)

> **Observación:** El dataset contiene 577 espacios patrimoniales y museos. La variable `Capacidad` tiene una distribución muy heterogénea (muchos valores en 0, probablemente «no especificado»). `Visita Guiada` y `Tienda` son indicadores binarios codificados como 0/10 en lugar del convencional 0/1.

## 2. Calidad del dataset — Valores nulos

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(1)
nulos_df = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': nulos_pct})
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(nulos_df.index, nulos_df['Porcentaje (%)'], color=PALETTE[4], edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('% de valores nulos')
ax.set_title('Completitud del dataset — Campos con valores nulos', fontweight='bold')
ax.set_xlim(0, 80)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(nulos_df.to_string())

> **Observación:** El campo **WEB** es el más incompleto (61%), lo que indica que una gran parte de los espacios patrimoniales carece de presencia web registrada, posiblemente por tratarse de patrimonio disperso o pequeños enclaves. Las valoraciones faltan en un 12% de los registros (espacios sin suficientes reseñas). La información geográfica y de nombre está completa al 100%.

## 3. Distribución por tipo de espacio

In [ ]:
tipo_counts = df['Tipo de lugar'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barplot
bars = axes[0].bar(tipo_counts.index, tipo_counts.values,
                   color=[PALETTE[0], PALETTE[1]], edgecolor='white', width=0.5)
axes[0].bar_label(bars, padding=4, fontsize=11)
axes[0].set_ylabel('Número de espacios')
axes[0].set_title('Espacios por tipo', fontweight='bold')
axes[0].set_ylim(0, 500)

# Pie
axes[1].pie(tipo_counts.values, labels=tipo_counts.index, autopct='%1.1f%%',
            colors=[PALETTE[0], PALETTE[1]], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción por tipo de lugar', fontweight='bold')

plt.tight_layout()
plt.show()

> **Observación:** El dataset se divide principalmente en dos categorías: **Patrimonio Cultural** (423 registros, 73%) y **Museos** (154, 27%). La categoría de patrimonio cultural engloba yacimientos arqueológicos, monumentos, iglesias, castillos y otras construcciones históricas diseminadas por el territorio.

## 4. Tipo de cultura — Especialización temática

In [ ]:
cultura_counts = df['Tipo de Cultura'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Horizontal bar
bars = axes[0].barh(cultura_counts.index[::-1], cultura_counts.values[::-1],
                    color=PALETTE[:len(cultura_counts)], edgecolor='white')
axes[0].bar_label(bars, padding=3, fontsize=9)
axes[0].set_xlabel('Número de espacios')
axes[0].set_title('Espacios por tipo de cultura', fontweight='bold')

# Solo museos por tipo de cultura
museos_cultura = df[df['Tipo de lugar'] == 'Museo']['Tipo de Cultura'].value_counts()
bars2 = axes[1].barh(museos_cultura.index[::-1], museos_cultura.values[::-1],
                     color=PALETTE[1], edgecolor='white')
axes[1].bar_label(bars2, padding=3, fontsize=9)
axes[1].set_xlabel('Número de museos')
axes[1].set_title('Museos por tipo de cultura', fontweight='bold')

plt.tight_layout()
plt.show()

> **Observación:** **Patrimonio Cultural** es la categoría dominante en ambas clasificaciones. Entre los museos destaca la variedad temática: historia, ciencias naturales, etnografía y arte. La categoría **Otros** es amplia, lo que sugiere oportunidad de mejorar la taxonomía del dataset. Destaca la presencia de museos de **Gastronomía** y **Marítimo**, reflejo de la identidad cultural vasca.

## 5. Distribución por provincia

In [ ]:
prov_tipo = df.groupby(['Provincia', 'Tipo de lugar']).size().unstack(fill_value=0)
prov_total = df['Provincia'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Stacked bar por tipo
prov_tipo.plot(kind='bar', ax=axes[0], color=[PALETTE[0], PALETTE[1]], edgecolor='white', width=0.6)
axes[0].set_xlabel('Provincia')
axes[0].set_ylabel('Número de espacios')
axes[0].set_title('Patrimonio y museos por provincia', fontweight='bold')
axes[0].legend(title='Tipo')
axes[0].tick_params(axis='x', rotation=0)

# Cultura por provincia
prov_cultura = df.groupby(['Provincia', 'Tipo de Cultura']).size().unstack(fill_value=0)
top_culturas = df['Tipo de Cultura'].value_counts().head(5).index
prov_cultura[top_culturas].plot(kind='bar', ax=axes[1], color=PALETTE[:5], edgecolor='white', width=0.7)
axes[1].set_xlabel('Provincia')
axes[1].set_ylabel('Número de espacios')
axes[1].set_title('Top 5 tipos de cultura por provincia', fontweight='bold')
axes[1].legend(title='Tipo de Cultura', fontsize=8, bbox_to_anchor=(1.01, 1))
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('\nTotal por provincia:')
print(prov_total.to_string())

> **Observación:** **Guipúzcoa** lidera también en patrimonio cultural (252 espacios), seguida de **Vizcaya** (228) y **Álava** (97). Álava tiene proporcionalmente más museos de historia, relacionados con su capitalidad administrativa en Vitoria-Gasteiz. Guipúzcoa y Vizcaya tienen más patrimonio disperso en el territorio, coherente con su mayor densidad urbana e industrial histórica.

## 6. Municipios con más espacios patrimoniales

In [ ]:
top_munic = df['Municipio'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_munic.index[::-1], top_munic.values[::-1],
               color=PALETTE[0], edgecolor='white')
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel('Número de espacios')
ax.set_title('Top 15 municipios con más espacios patrimoniales', fontweight='bold')
plt.tight_layout()
plt.show()

> **Observación:** **Vitoria-Gasteiz** destaca muy por encima del resto como municipio con mayor densidad de espacios patrimoniales, gracias a su casco medieval bien conservado y la concentración de museos institucionales. **Bilbao** y **San Sebastián** siguen a distancia como grandes centros urbanos. La presencia de municipios pequeños como Oñati o Segura refleja el rico patrimonio histórico disperso por la Euskadi rural.

## 7. Servicios: Visita guiada y tienda

In [ ]:
# Recodificar 0/10 a 0/1
df['visita_bin'] = (df['Visita Guiada'] > 0).astype(int)
df['tienda_bin'] = (df['Tienda'] > 0).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Visita guiada global
vg = df['visita_bin'].value_counts().rename({0: 'Sin visita guiada', 1: 'Con visita guiada'})
axes[0].pie(vg.values, labels=vg.index, autopct='%1.1f%%',
            colors=['#BDC3C7', PALETTE[0]], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('¿Ofrece visita guiada?', fontweight='bold')

# Tienda global
ti = df['tienda_bin'].value_counts().rename({0: 'Sin tienda', 1: 'Con tienda'})
axes[1].pie(ti.values, labels=ti.index, autopct='%1.1f%%',
            colors=['#BDC3C7', PALETTE[1]], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('¿Tiene tienda?', fontweight='bold')

# Visita guiada por tipo
vg_tipo = df.groupby('Tipo de lugar')['visita_bin'].mean() * 100
axes[2].bar(vg_tipo.index, vg_tipo.values, color=[PALETTE[0], PALETTE[1]], edgecolor='white')
for i, v in enumerate(vg_tipo.values):
    axes[2].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[2].set_ylabel('% con visita guiada')
axes[2].set_title('Visita guiada por tipo de lugar', fontweight='bold')
axes[2].set_ylim(0, 80)

plt.suptitle('Servicios disponibles en los espacios patrimoniales', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Espacios con visita guiada: {df["visita_bin"].sum()} ({df["visita_bin"].mean()*100:.1f}%)')
print(f'Espacios con tienda: {df["tienda_bin"].sum()} ({df["tienda_bin"].mean()*100:.1f}%)')

> **Observación:** Aproximadamente la **mitad de los espacios** ofrece visita guiada y solo el **20% tiene tienda**. Los **Museos** tienen una tasa de visita guiada significativamente mayor que los espacios de patrimonio cultural, lo que tiene sentido dado su carácter más institucional y orientado al público visitante.

## 8. Análisis de capacidad

In [ ]:
# Excluir capacidad 0 (no especificada)
cap_valida = df[df['Capacidad'] > 0]['Capacidad']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histograma de capacidad
axes[0].hist(cap_valida, bins=30, color=PALETTE[2], edgecolor='white')
axes[0].axvline(cap_valida.mean(), color='red', linestyle='--', label=f'Media: {cap_valida.mean():.0f}')
axes[0].axvline(cap_valida.median(), color='orange', linestyle='--', label=f'Mediana: {cap_valida.median():.0f}')
axes[0].set_xlabel('Capacidad')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title(f'Distribución de capacidad\n(sobre {len(cap_valida)} espacios con dato)', fontweight='bold')
axes[0].legend()

# Capacidad media por tipo
cap_tipo = df[df['Capacidad'] > 0].groupby('Tipo de lugar')['Capacidad'].mean()
axes[1].bar(cap_tipo.index, cap_tipo.values, color=[PALETTE[0], PALETTE[1]], edgecolor='white')
for i, v in enumerate(cap_tipo.values):
    axes[1].text(i, v + 20, f'{v:.0f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Capacidad media')
axes[1].set_title('Capacidad media por tipo de lugar', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Espacios con capacidad especificada: {len(cap_valida)} ({len(cap_valida)/len(df)*100:.1f}%)')
print(f'Capacidad media: {cap_valida.mean():.0f} | Mediana: {cap_valida.median():.0f} | Máx: {cap_valida.max():.0f}')

> **Observación:** La capacidad es muy variable entre espacios. La distribución es muy asimétrica: la mayoría tiene capacidades pequeñas/medianas, pero algunos enclaves (grandes museos, recintos monumentales) presentan capacidades muy elevadas. Los **Museos** tienen en media mayor capacidad que el patrimonio cultural, coherente con su diseño como equipamientos de acceso masivo.

## 9. Análisis de valoraciones

In [ ]:
df_val = df.dropna(subset=['valoracion', 'num_resenas'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribución de valoraciones
axes[0].hist(df_val['valoracion'], bins=20, color=PALETTE[0], edgecolor='white')
axes[0].axvline(df_val['valoracion'].mean(), color='red', linestyle='--', label=f'Media: {df_val["valoracion"].mean():.2f}')
axes[0].axvline(df_val['valoracion'].median(), color='orange', linestyle='--', label=f'Mediana: {df_val["valoracion"].median():.2f}')
axes[0].set_xlabel('Valoración')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de valoraciones', fontweight='bold')
axes[0].legend(fontsize=9)

# Boxplot por tipo
df_val.boxplot(column='valoracion', by='Tipo de lugar', ax=axes[1], 
               boxprops=dict(color=PALETTE[0]), medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Valoración por tipo de lugar', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Valoración')
plt.sca(axes[1])
plt.title('Valoración por tipo de lugar')

# Valoración media por tipo de cultura (top 8)
val_cultura = df_val.groupby('Tipo de Cultura')['valoracion'].mean().sort_values(ascending=True)
bars = axes[2].barh(val_cultura.index, val_cultura.values, color=PALETTE[:len(val_cultura)], edgecolor='white')
axes[2].set_xlim(3.5, 5)
axes[2].set_xlabel('Valoración media')
axes[2].set_title('Valoración media por tipo de cultura', fontweight='bold')
for bar, val in zip(bars, val_cultura.values):
    axes[2].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Valoración media global: {df_val["valoracion"].mean():.2f}')
print(f'Mediana de reseñas: {df_val["num_resenas"].median():.0f}')

> **Observación:** Las valoraciones del patrimonio cultural son también muy positivas, con una media superior a 4.0. Los **Museos** tienden a tener valoraciones más consistentes (menos variabilidad) que el patrimonio cultural, posiblemente porque cuentan con más infraestructura de atención al visitante. Los espacios de **Arte y Etnografía** logran notas especialmente altas.

## 10. Valoración vs. servicios disponibles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, label, color_on, color_off in [
    (axes[0], 'visita_bin', 'Visita Guiada', PALETTE[0], '#BDC3C7'),
    (axes[1], 'tienda_bin', 'Tienda', PALETTE[1], '#BDC3C7')
]:
    data_no = df_val[df_val[col] == 0]['valoracion']
    data_si = df_val[df_val[col] == 1]['valoracion']
    bp = ax.boxplot([data_no, data_si], labels=[f'Sin {label}', f'Con {label}'],
                    patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor(color_off)
    bp['boxes'][1].set_facecolor(color_on)
    ax.set_ylabel('Valoración')
    ax.set_title(f'Valoración según {label}', fontweight='bold')
    ax.set_ylim(1.5, 5.5)
    ax.text(1, 5.2, f'μ={data_no.mean():.2f}', ha='center', fontsize=10, color='gray')
    ax.text(2, 5.2, f'μ={data_si.mean():.2f}', ha='center', fontsize=10, color=color_on)

plt.suptitle('¿Impactan los servicios en la valoración?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Observación:** Los espacios que ofrecen **visita guiada o tienda** obtienen valoraciones ligeramente superiores. Esto sugiere que una mayor inversión en servicios al visitante se traduce en una mejor experiencia percibida. No obstante, la diferencia no es drástica, lo que indica que la calidad intrínseca del patrimonio sigue siendo el factor dominante.

## 11. Mapa de calor: correlación entre variables numéricas

In [ ]:
num_cols = ['valoracion', 'num_resenas', 'Capacidad', 'visita_bin', 'tienda_bin', 'Patrocinado']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlación entre variables numéricas', fontweight='bold')
plt.tight_layout()
plt.show()

> **Observación:** La correlación más relevante es la positiva entre **Capacidad y número de reseñas**: los espacios más grandes atraen más visitantes y generan más valoraciones. La correlación entre visita guiada y tienda es moderada, indicando que suelen ir juntos como parte de una oferta más completa. La valoración no correlaciona fuertemente con ninguna otra variable cuantitativa, confirmando que es un indicador de calidad percibida independiente del tamaño o los servicios.

## 12. Distribución geográfica

In [ ]:
prov_colors = {'Guipúzcoa': '#C0392B', 'Vizcaya': '#2E86C1', 'Álava': '#27AE60'}
tipo_markers = {'Museo': '*', 'Patrimonio cultural': 'o'}

fig, ax = plt.subplots(figsize=(12, 6))

for tipo, marker in tipo_markers.items():
    for prov, color in prov_colors.items():
        subset = df[(df['Provincia'] == prov) & (df['Tipo de lugar'] == tipo)]
        size = 80 if tipo == 'Museo' else 25
        alpha = 0.8 if tipo == 'Museo' else 0.5
        ax.scatter(subset['lon'], subset['lat'], c=color, alpha=alpha, s=size, marker=marker)

# Leyenda manual
import matplotlib.lines as mlines
handles = []
for prov, color in prov_colors.items():
    handles.append(mlines.Line2D([], [], color=color, marker='o', linestyle='None', markersize=7, label=prov))
handles.append(mlines.Line2D([], [], color='gray', marker='*', linestyle='None', markersize=10, label='Museo'))
handles.append(mlines.Line2D([], [], color='gray', marker='o', linestyle='None', markersize=6, label='Patrimonio cultural'))

ax.legend(handles=handles, loc='upper left', fontsize=9)
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_title('Distribución geográfica del patrimonio cultural en Euskadi', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Observación:** El patrimonio cultural está ampliamente distribuido por todo el territorio vasco, con una concentración visible en el eje costero (Bilbao–San Sebastián) y en la capital alavesa (Vitoria-Gasteiz). Los **Museos** (estrellas) se agrupan en los núcleos urbanos principales, mientras que el **Patrimonio Cultural** (círculos) aparece disperso también en zonas rurales e interiores, especialmente en el interior de Guipúzcoa y Álava.

## 13. Top 10 — Espacios más reseñados

In [ ]:
top10 = (
    df.dropna(subset=['num_resenas'])
    .sort_values('num_resenas', ascending=False)
    .head(10)[['Nombre', 'Municipio', 'Tipo de lugar', 'Tipo de Cultura', 'valoracion', 'num_resenas']]
    .reset_index(drop=True)
)
top10.index += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Más reseñas
bars = axes[0].barh(top10['Nombre'][::-1], top10['num_resenas'][::-1], color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Número de reseñas')
axes[0].set_title('Top 10 espacios con más reseñas', fontweight='bold')

# Mejor valorados con >200 reseñas
top10_val = (
    df[df['num_resenas'] >= 200]
    .dropna(subset=['valoracion'])
    .sort_values('valoracion', ascending=False)
    .head(10)[['Nombre', 'valoracion']]
    .reset_index(drop=True)
)
bars2 = axes[1].barh(top10_val['Nombre'][::-1], top10_val['valoracion'][::-1], color=PALETTE[1], edgecolor='white')
axes[1].set_xlim(4.4, 5.0)
axes[1].set_xlabel('Valoración media')
axes[1].set_title('Top 10 mejor valorados (≥200 reseñas)', fontweight='bold')
for bar, val in zip(bars2, top10_val['valoracion'][::-1]):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.1f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print(top10.to_string())

> **Observación:** Los espacios con más reseñas son los grandes iconos turísticos vascos (Guggenheim, museos nacionales, catedrales). Los espacios mejor valorados con volumen significativo de reseñas suelen ser museos especializados y pequeños tesoros patrimoniales con una audiencia muy fiel y satisfecha.

---
## 📝 Resumen ejecutivo

| Dimensión | Hallazgo clave |
|---|---|
| **Volumen** | 577 espacios patrimoniales registrados |
| **Tipo dominante** | Patrimonio Cultural (73%), Museos (27%) |
| **Provincia líder** | Guipúzcoa (44%), seguida de Vizcaya |
| **Municipio destacado** | Vitoria-Gasteiz concentra más espacios |
| **Valoración media** | ~4.3/5 — alta satisfacción del visitante |
| **Visita guiada** | ~50% de los espacios la ofrecen |
| **Tienda** | Solo ~20% dispone de tienda |
| **Dato incompleto** | WEB falta en 61% de registros |
| **Geografía** | Patrimonio disperso + museos en capitales |